# Sağlık Sigortası Prim Tahmini

Bu projede yaş, bmi, sigara gibi bilgilere bakarak sigorta primini tahmin edeceğim.


In [ ]:
import pandas as pd
pd.set_option('display.max_columns',100)
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns


### Data


In [ ]:
df=pd.read_csv('data/insurance.csv')
df.head()


### EDA


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


In [ ]:
df['smoker'].value_counts()


### Görselleştirme


In [ ]:
sns.boxplot(x='smoker',y='charges',data=df)
plt.show()


In [ ]:
sns.scatterplot(x='age',y='charges',hue='smoker',data=df)
plt.show()


### Boş veri


In [ ]:
df['bmi']=df['bmi'].fillna(df['bmi'].median())
df['age']=df['age'].fillna(df['age'].median())


### Feature Engineering


In [ ]:
df['obez']=(df['bmi']>=30).astype(int)
x=df.drop('charges',axis=1)
y=df['charges']
x=pd.get_dummies(x,drop_first=True)


### Train Test Split


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


### 3 Model


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor,RandomForestRegressor
from sklearn.metrics import r2_score,mean_absolute_error

modeller=[]
for ad,m in [('Linear',LinearRegression()),('GBM',GradientBoostingRegressor(random_state=42)),('RF',RandomForestRegressor(random_state=42))]:
    m.fit(x_train,y_train)
    p=m.predict(x_test)
    print(ad,'R2',round(r2_score(y_test,p),3),'MAE',round(mean_absolute_error(y_test,p),1))
    modeller.append((ad,m,r2_score(y_test,p)))


### Feature Importance + Residual


In [ ]:
best=max(modeller,key=lambda t:t[2])[1]
pred=best.predict(x_test)

if hasattr(best,'feature_importances_'):
    s=pd.Series(best.feature_importances_,index=x.columns).sort_values(ascending=False)
    s.head(8).plot(kind='barh')
    plt.show()

plt.scatter(pred,y_test-pred)
plt.axhline(0,color='red')
plt.show()


In [ ]:
import joblib
joblib.dump(best,'../../models/regression_health_insurance.joblib')


### Sonuç

Sigara içmek primi ciddi artırıyor, bu beklediğim gibiydi. GBM / RF linear'dan iyi. Hedefi tutturdum, prim tahmini kullanılabilir seviyede.
